# ProtMamba Nonlinear Probe Validation: Linear vs. MLP Frozen Head

**Core Question**: Is ProtMamba's chance-level linear probe accuracy evidence of
*rich nonlinear encoding* or *uninformative representations*?

## The Problem

In the Ghost Detection experiment, ProtMamba's linear frozen head achieved ~50%
accuracy (random chance) at **all** sequence lengths, including the training length (L=100).
We interpreted this as "resistance to static linearization" -- the claim being that
ProtMamba encodes information as deep nonlinear trajectories rather than the brittle
linear "Lego-block" formats that discrete models use.

A key question is: **How do you distinguish "rich nonlinear representation"
from "uninformative representation"?**

## The Control Experiment

We run the **identical** frozen head protocol from `ProtMamba_Ghost_Detection.ipynb`,
but with **two classifiers side-by-side**:

1. **Linear Probe** (LogisticRegression) -- the original test
2. **MLP Probe** (2-layer neural network) -- the nonlinear control

## Methodology

- **Task**: E. coli vs. Human protein classification (identical to Ghost Detection)
- **Signal**: 50 aa real protein region (first 50 residues)
- **Padding**: Random amino acids to target lengths [100, 200, 400, 800]
- **Pooling**: LOCAL (signal region only) + GLOBAL (full sequence)
- **Probes**: LogisticRegression vs. MLPClassifier (2 hidden layers, 256/64 units)
- **Evaluation**: 5-fold stratified cross-validation
- **Seed**: 320 (consistent with all prior experiments)

---


In [ ]:
# Install Dependencies

print('Installing dependencies...')
!pip install -q torch transformers matplotlib seaborn pandas scipy scikit-learn einops biopython
!pip install -q causal-conv1d>=1.1.0
!pip install -q mamba-ssm

import os
if not os.path.exists('ProtMamba-ssm'):
    !git clone https://github.com/Bitbol-Lab/ProtMamba-ssm.git
!pip install -q -e ProtMamba-ssm/

WEIGHTS_DIR = 'protmamba_weights'
if not os.path.exists(WEIGHTS_DIR):
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    !wget -q -O protmamba_weights.zip https://github.com/Bitbol-Lab/ProtMamba-ssm/releases/download/v1.0/ProtMamba_model-weights.zip
    !unzip -q -o protmamba_weights.zip -d {WEIGHTS_DIR}
    !rm protmamba_weights.zip

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import orthogonal_procrustes
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import seaborn as sns

matplotlib.rcParams.update({'font.size': 11})
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')


In [ ]:
# Configuration

# --- Paths ---
GHOST_CACHE = './results/protmamba_ghost_detection/cache'

OUTPUT_BASE = './results/protmamba_nonlinear_probe/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Frozen Head config (identical to Ghost Detection)
N_CLASSIFICATION_SEQS = 200      # per class (200 ecoli + 200 human = 400 total)
SIGNAL_LENGTH = 50               # 50 aa signal region
TARGET_LENGTHS = [100, 200, 400, 800]  # total sequence lengths to test
N_CV_FOLDS = 5
RANDOM_SEED = 320

# MLP Probe config
MLP_HIDDEN_LAYERS = (256, 64)    # 2 hidden layers
MLP_MAX_ITER = 500
MLP_LEARNING_RATE_INIT = 1e-3

ghost_cache_available = os.path.exists(GHOST_CACHE)
print(f'Output: {RESULTS_DIR}')
print(f'Ghost Detection cache: {GHOST_CACHE} (available: {ghost_cache_available})')
print(f'MLP architecture: input -> {MLP_HIDDEN_LAYERS[0]} -> {MLP_HIDDEN_LAYERS[1]} -> 2')
print(f'Signal: {SIGNAL_LENGTH} aa | Lengths: {TARGET_LENGTHS}')


In [ ]:
# ProtMamba Model Loader
#
# Same embedding function as used in the Ghost Detection experiment.
# Only needed if cached embeddings are not available.

import gc, glob, time


def find_checkpoint_path(weights_dir='protmamba_weights'):
    for pattern in [
        f'{weights_dir}/**/checkpoint*',
        f'{weights_dir}/**/model*.pt',
        f'{weights_dir}/**/pytorch_model.bin',
        f'{weights_dir}/**/*.pt',
    ]:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            path = matches[0]
            return os.path.dirname(path) if os.path.isfile(path) else path
    return weights_dir


def make_protmamba_embedding_fn(batch_size=8, signal_length=None):
    import sys; sys.path.append(os.path.abspath('ProtMamba-ssm'))
    from ProtMamba_ssm.modules import MambaLMHeadModelwithPosids, load_model
    from ProtMamba_ssm.utils import AA_TO_ID

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Loading ProtMamba on {device}...')

    ckpt_path = find_checkpoint_path()
    model = load_model(
        ckpt_path, model_class=MambaLMHeadModelwithPosids,
        device=device, dtype=torch.float16, checkpoint_mixer=False,
    )
    model.eval()

    n_layers = model.config.n_layer
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Params: {n_params:.1f}M | Layers: {n_layers}')

    aa_to_id = AA_TO_ID.copy()
    unk_id = aa_to_id.get('X', aa_to_id.get('<unk>', 0))

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1
            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f'    Batch {batch_idx}/{n_batches}', end='\r')
            max_len = max(len(s) for s in batch_seqs)
            token_ids = []
            for seq in batch_seqs:
                ids = [aa_to_id.get(aa, unk_id) for aa in seq]
                ids += [0] * (max_len - len(ids))
                token_ids.append(ids)
            tokens = torch.tensor(token_ids, dtype=torch.long, device=device)
            pos_ids = torch.arange(max_len, device=device).unsqueeze(0).expand(len(batch_seqs), -1)
            hidden_states = model(
                input_ids=tokens, save_layer=[n_layers], position_ids=pos_ids,
            )
            last_hidden = hidden_states[n_layers]
            batch_embeds = []
            for j, seq in enumerate(batch_seqs):
                if signal_length is not None and signal_length < len(seq):
                    pooled = last_hidden[j, :signal_length, :].mean(axis=0)
                else:
                    pooled = last_hidden[j, :len(seq), :].mean(axis=0)
                batch_embeds.append(pooled.cpu().float().numpy())
            embeddings.extend(batch_embeds)

        return np.array(embeddings)

    return embed


In [ ]:
# Download Classification Protein Sequences
#
# Same sequence loader as used in the Ghost Detection experiment.
# Task: Distinguish real E. coli proteins from real Human proteins.

import urllib.request

AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

ECOLI_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:83333)+AND+(reviewed:true)&size=500'
HUMAN_URL = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(organism_id:9606)+AND+(reviewed:true)&size=500'


def download_protein_sequences(url, n_sequences, signal_length, seed, cache_tag):
    cache_file = f'{CACHE_DIR}/{cache_tag}_{n_sequences}_{signal_length}.txt'
    if os.path.exists(cache_file):
        print(f'  Loading cached: {cache_file}')
        with open(cache_file) as f:
            return [line.strip() for line in f if line.strip()]

    print(f'  Downloading from UniProt...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Python/alphagenome'})
    with urllib.request.urlopen(req) as response:
        raw = response.read().decode('ascii')

    proteins = []
    current_seq = []
    for line in raw.split('\n'):
        if line.startswith('>'):
            if current_seq:
                proteins.append(''.join(current_seq))
            current_seq = []
        elif line.strip():
            current_seq.append(line.strip().upper())
    if current_seq:
        proteins.append(''.join(current_seq))

    print(f'  Parsed {len(proteins)} proteins')

    valid_aa = set(AMINO_ACIDS)
    valid_proteins = [p for p in proteins if len(p) >= signal_length
                      and all(c in valid_aa for c in p[:signal_length])]
    print(f'  Valid (>={signal_length} aa, standard AAs): {len(valid_proteins)}')

    rng = np.random.default_rng(seed)
    if len(valid_proteins) > n_sequences:
        idx = rng.choice(len(valid_proteins), n_sequences, replace=False)
        valid_proteins = [valid_proteins[i] for i in idx]

    signals = [p[:signal_length] for p in valid_proteins[:n_sequences]]

    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(cache_file, 'w') as f:
        f.write('\n'.join(signals))
    print(f'  Extracted {len(signals)} signal sequences ({signal_length} aa each)')
    return signals


def pad_protein_sequence(signal_seq, target_length, rng):
    pad_needed = target_length - len(signal_seq)
    if pad_needed <= 0:
        return signal_seq[:target_length]
    pad = ''.join(rng.choice(list(AMINO_ACIDS), size=pad_needed))
    return signal_seq + pad


# Download sequences
print('Downloading E. coli proteins...')
ecoli_signals = download_protein_sequences(
    ECOLI_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, RANDOM_SEED, 'ecoli')

print('\nDownloading Human proteins...')
human_signals = download_protein_sequences(
    HUMAN_URL, N_CLASSIFICATION_SEQS, SIGNAL_LENGTH, RANDOM_SEED, 'human')

all_signals = ecoli_signals + human_signals
labels = np.array([0] * len(ecoli_signals) + [1] * len(human_signals))
print(f'\nTotal: {len(all_signals)} sequences ({len(ecoli_signals)} E.coli + {len(human_signals)} Human)')


In [ ]:
# Embed Padded Sequences (or Load from Ghost Detection Cache)

frozen_head_embeddings_local = {}
frozen_head_embeddings_global = {}

needs_inference = False
for target_len in TARGET_LENGTHS:
    local_path = None
    for cache_root in [GHOST_CACHE, CACHE_DIR]:
        candidate = f'{cache_root}/frozen_head_L{target_len}.npy'
        if os.path.exists(candidate):
            local_path = candidate
            break

    global_path = None
    for cache_root in [GHOST_CACHE, CACHE_DIR]:
        candidate = f'{cache_root}/frozen_head_global_L{target_len}.npy'
        if os.path.exists(candidate):
            global_path = candidate
            break

    if local_path:
        print(f'L={target_len}: Loading LOCAL from {local_path}')
        frozen_head_embeddings_local[target_len] = np.load(local_path)
    else:
        needs_inference = True

    if global_path:
        print(f'L={target_len}: Loading GLOBAL from {global_path}')
        frozen_head_embeddings_global[target_len] = np.load(global_path)
    else:
        needs_inference = True

if needs_inference:
    print('\nSome embeddings not cached. Running live inference...')
    embed_fn_local = make_protmamba_embedding_fn(batch_size=8, signal_length=SIGNAL_LENGTH)
    embed_fn_global = make_protmamba_embedding_fn(batch_size=8, signal_length=None)
    rng_pad = np.random.default_rng(RANDOM_SEED)

    for target_len in TARGET_LENGTHS:
        if target_len not in frozen_head_embeddings_local:
            print(f'\nL={target_len}: Padding + Embedding')

            padded = [pad_protein_sequence(s, target_len, rng_pad) for s in all_signals]
            print(f'  Signal fraction: {SIGNAL_LENGTH / target_len * 100:.1f}%')

            print(f'  LOCAL POOL (first {SIGNAL_LENGTH} tokens)...')
            embs_local = embed_fn_local(padded)
            embs_local = np.nan_to_num(embs_local, nan=0.0, posinf=1e6, neginf=-1e6)
            save_path = f'{CACHE_DIR}/frozen_head_L{target_len}.npy'
            np.save(save_path, embs_local)
            frozen_head_embeddings_local[target_len] = embs_local
            print(f'  Saved: {save_path} | Shape: {embs_local.shape}')

        if target_len not in frozen_head_embeddings_global:
            rng_pad_g = np.random.default_rng(RANDOM_SEED)
            padded_g = [pad_protein_sequence(s, target_len, rng_pad_g) for s in all_signals]

            print(f'  GLOBAL POOL (all {target_len} tokens)...')
            embs_global = embed_fn_global(padded_g)
            embs_global = np.nan_to_num(embs_global, nan=0.0, posinf=1e6, neginf=-1e6)
            save_path = f'{CACHE_DIR}/frozen_head_global_L{target_len}.npy'
            np.save(save_path, embs_global)
            frozen_head_embeddings_global[target_len] = embs_global
            print(f'  Saved: {save_path} | Shape: {embs_global.shape}')

    del embed_fn_local, embed_fn_global
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
else:
    print('\nAll embeddings loaded from cache. No inference needed.')

print(f'\nLocal embeddings: {sorted(frozen_head_embeddings_local.keys())}')
print(f'Global embeddings: {sorted(frozen_head_embeddings_global.keys())}')


In [ ]:
# LINEAR vs. MLP FROZEN HEAD -- Linear vs. MLP Frozen Head Comparison
#
# For each target length and pooling mode:
#   1. Train LogisticRegression (linear probe)
#   2. Train MLPClassifier (nonlinear probe)
#   3. Train wider MLPClassifier (robustness check)

print('=' * 70)
print('NONLINEAR PROBE VALIDATION: Linear vs. MLP Frozen Head')
print('=' * 70)

probe_rows = []


def run_probe_comparison(X, y, target_len, pool_mode, seed=RANDOM_SEED):
    cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=seed)

    # Linear Probe
    clf_linear = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=seed)
    )
    scores_linear = cross_val_score(clf_linear, X, y, cv=cv, scoring='accuracy')

    # MLP Probe (256, 64)
    clf_mlp = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN_LAYERS,
            max_iter=MLP_MAX_ITER,
            learning_rate_init=MLP_LEARNING_RATE_INIT,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=seed,
            activation='relu',
        )
    )
    scores_mlp = cross_val_score(clf_mlp, X, y, cv=cv, scoring='accuracy')

    # MLP Probe Wide (512, 256, 64)
    clf_mlp_wide = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(512, 256, 64),
            max_iter=MLP_MAX_ITER,
            learning_rate_init=MLP_LEARNING_RATE_INIT,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=seed,
            activation='relu',
        )
    )
    scores_mlp_wide = cross_val_score(clf_mlp_wide, X, y, cv=cv, scoring='accuracy')

    return {
        'target_length': target_len,
        'pool_mode': pool_mode,
        'signal_fraction': SIGNAL_LENGTH / target_len,
        'linear_mean': scores_linear.mean(),
        'linear_std': scores_linear.std(),
        'linear_folds': scores_linear.tolist(),
        'mlp_mean': scores_mlp.mean(),
        'mlp_std': scores_mlp.std(),
        'mlp_folds': scores_mlp.tolist(),
        'mlp_wide_mean': scores_mlp_wide.mean(),
        'mlp_wide_std': scores_mlp_wide.std(),
        'mlp_wide_folds': scores_mlp_wide.tolist(),
        'mlp_minus_linear': scores_mlp.mean() - scores_linear.mean(),
        'mlp_wide_minus_linear': scores_mlp_wide.mean() - scores_linear.mean(),
    }


# --- LOCAL POOLING ---
print('\n--- LOCAL POOLING (signal region only) ---')
for target_len in TARGET_LENGTHS:
    if target_len not in frozen_head_embeddings_local:
        print(f'  L={target_len}: SKIPPED (no embeddings)')
        continue
    X = frozen_head_embeddings_local[target_len]
    y = labels[:X.shape[0]]
    print(f'\n  L={target_len} | Shape: {X.shape} | Signal: {SIGNAL_LENGTH/target_len*100:.0f}%')

    result = run_probe_comparison(X, y, target_len, 'local')
    probe_rows.append(result)

    print(f'    Linear:          {result["linear_mean"]:.4f} +/- {result["linear_std"]:.4f}')
    print(f'    MLP (256,64):    {result["mlp_mean"]:.4f} +/- {result["mlp_std"]:.4f}  '
          f'(delta: {result["mlp_minus_linear"]:+.4f})')
    print(f'    MLP (512,256,64):{result["mlp_wide_mean"]:.4f} +/- {result["mlp_wide_std"]:.4f}  '
          f'(delta: {result["mlp_wide_minus_linear"]:+.4f})')

# --- GLOBAL POOLING ---
print('\n\n--- GLOBAL POOLING (full sequence) ---')
for target_len in TARGET_LENGTHS:
    if target_len not in frozen_head_embeddings_global:
        print(f'  L={target_len}: SKIPPED (no embeddings)')
        continue
    X = frozen_head_embeddings_global[target_len]
    y = labels[:X.shape[0]]
    print(f'\n  L={target_len} | Shape: {X.shape} | Signal: {SIGNAL_LENGTH/target_len*100:.0f}%')

    result = run_probe_comparison(X, y, target_len, 'global')
    probe_rows.append(result)

    print(f'    Linear:          {result["linear_mean"]:.4f} +/- {result["linear_std"]:.4f}')
    print(f'    MLP (256,64):    {result["mlp_mean"]:.4f} +/- {result["mlp_std"]:.4f}  '
          f'(delta: {result["mlp_minus_linear"]:+.4f})')
    print(f'    MLP (512,256,64):{result["mlp_wide_mean"]:.4f} +/- {result["mlp_wide_std"]:.4f}  '
          f'(delta: {result["mlp_wide_minus_linear"]:+.4f})')

df_probes = pd.DataFrame(probe_rows)
df_probes.to_csv(f'{RESULTS_DIR}/nonlinear_probe_results.csv', index=False)
print(f'\nSaved: {RESULTS_DIR}/nonlinear_probe_results.csv')


In [ ]:
# Diagnostic Verdict

print('=' * 70)
print('DIAGNOSTIC VERDICT')
print('=' * 70)

CHANCE = 0.50
MEANINGFUL_THRESHOLD = 0.55

for pool_mode in ['local', 'global']:
    df_pool = df_probes[df_probes['pool_mode'] == pool_mode]
    if df_pool.empty:
        continue

    print(f'\n--- {pool_mode.upper()} POOLING ---')
    for _, row in df_pool.iterrows():
        L = int(row['target_length'])
        lin = row['linear_mean']
        mlp = row['mlp_mean']
        mlp_w = row['mlp_wide_mean']
        best_mlp = max(mlp, mlp_w)
        gap = best_mlp - lin

        print(f'\n  L={L} (signal: {row["signal_fraction"]*100:.0f}%)')
        print(f'    Linear: {lin:.4f} | MLP: {mlp:.4f} | MLP-wide: {mlp_w:.4f} | Gap: {gap:+.4f}')

        if lin > MEANINGFUL_THRESHOLD and best_mlp > MEANINGFUL_THRESHOLD:
            if gap < 0.02:
                print(f'    VERDICT: LINEAR ENCODING. Both probes succeed.')
            else:
                print(f'    VERDICT: MIXED. Both succeed, but MLP is better.')
        elif lin <= MEANINGFUL_THRESHOLD and best_mlp > MEANINGFUL_THRESHOLD:
            print(f'    VERDICT: NONLINEAR ENCODING CONFIRMED.')
            print(f'    MLP recovers signal (acc={best_mlp:.3f}) that linear cannot (acc={lin:.3f}).')
        elif lin <= MEANINGFUL_THRESHOLD and best_mlp <= MEANINGFUL_THRESHOLD:
            print(f'    VERDICT: UNINFORMATIVE REPRESENTATION.')
            print(f'    Neither probe recovers signal.')
        else:
            print(f'    VERDICT: ANOMALOUS.')

# Overall
print(f'\n{"=" * 70}')
print('OVERALL ASSESSMENT')
print('=' * 70)

df_local = df_probes[df_probes['pool_mode'] == 'local']
if not df_local.empty:
    avg_linear = df_local['linear_mean'].mean()
    avg_mlp = df_local[['mlp_mean', 'mlp_wide_mean']].max(axis=1).mean()
    avg_gap = avg_mlp - avg_linear

    if avg_linear <= MEANINGFUL_THRESHOLD and avg_mlp > MEANINGFUL_THRESHOLD:
        print('\nPROTMAMBA NONLINEAR ENCODING: VALIDATED')
        print(f'  Average linear acc: {avg_linear:.4f} (at chance)')
        print(f'  Average best MLP acc: {avg_mlp:.4f} (above chance)')
        print(f'  Average gap: {avg_gap:+.4f}')
        print('\n  ProtMamba encodes biological information in a nonlinear')
        print('  geometric format. The "resistance to linearization" claim is CONFIRMED.')
    elif avg_linear <= MEANINGFUL_THRESHOLD and avg_mlp <= MEANINGFUL_THRESHOLD:
        print('\nPROTMAMBA ENCODING: UNINFORMATIVE')
        print(f'  Average linear acc: {avg_linear:.4f} (at chance)')
        print(f'  Average best MLP acc: {avg_mlp:.4f} (at chance)')
        print('\n  Neither linear nor nonlinear probes extract species identity.')
        print('  The "resistance to linearization" framing must be revised.')
    else:
        print(f'\n  Average linear acc: {avg_linear:.4f}')
        print(f'  Average best MLP acc: {avg_mlp:.4f}')
        print(f'  Average gap: {avg_gap:+.4f}')
        print('  See per-length verdicts above.')


In [ ]:
# Visualization -- Linear vs. MLP Comparison

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

C_LINEAR = '#6B7280'
C_MLP = '#2563EB'
C_MLP_W = '#7C3AED'
C_CHANCE = '#DC2626'

for ax_idx, pool_mode in enumerate(['local', 'global']):
    ax = axes[ax_idx]
    df_pool = df_probes[df_probes['pool_mode'] == pool_mode]
    if df_pool.empty:
        ax.text(0.5, 0.5, f'No {pool_mode} data', ha='center', va='center',
                transform=ax.transAxes, fontsize=14, color='gray')
        continue

    x = df_pool['target_length'].values
    width = np.min(np.diff(x)) * 0.25 if len(x) > 1 else 20

    ax.bar(x - width, df_pool['linear_mean'], width, yerr=df_pool['linear_std'],
           color=C_LINEAR, alpha=0.85, capsize=4, label='Linear Probe', edgecolor='white')
    ax.bar(x, df_pool['mlp_mean'], width, yerr=df_pool['mlp_std'],
           color=C_MLP, alpha=0.85, capsize=4, label='MLP (256,64)', edgecolor='white')
    ax.bar(x + width, df_pool['mlp_wide_mean'], width, yerr=df_pool['mlp_wide_std'],
           color=C_MLP_W, alpha=0.85, capsize=4, label='MLP (512,256,64)', edgecolor='white')

    ax.axhline(y=0.5, color=C_CHANCE, linestyle='--', linewidth=1.5, alpha=0.7, label='Chance')
    ax.set_xlabel('Sequence Length (aa)', fontsize=11)
    ax.set_ylabel('Frozen Head Accuracy', fontsize=11)
    title_label = 'A' if ax_idx == 0 else 'B'
    ax.set_title(f'{title_label}. {pool_mode.title()} Pooling: Linear vs. MLP',
                 fontweight='bold', fontsize=12)
    ax.set_ylim(0.35, 1.05)
    ax.set_xticks(x)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.15, axis='y')

    for _, row in df_pool.iterrows():
        ax.annotate(f'{row["signal_fraction"]*100:.0f}% sig',
                    xy=(row['target_length'], 0.38),
                    fontsize=7, color='#9CA3AF', ha='center')

# Panel C: Gap plot
ax = axes[2]
for pool_mode, color, marker in [('local', C_MLP, 'o'), ('global', '#10B981', 's')]:
    df_pool = df_probes[df_probes['pool_mode'] == pool_mode]
    if df_pool.empty:
        continue
    best_mlp = df_pool[['mlp_mean', 'mlp_wide_mean']].max(axis=1)
    gap = best_mlp.values - df_pool['linear_mean'].values
    ax.plot(df_pool['target_length'], gap, marker=marker, linewidth=2.5,
            markersize=10, color=color, label=f'{pool_mode.title()} pool')

ax.axhline(y=0, color=C_LINEAR, linestyle=':', linewidth=1.5, alpha=0.7)
ax.axhline(y=0.10, color='#F59E0B', linestyle='--', linewidth=1, alpha=0.5,
           label='Meaningful gap (10%)')
ax.set_xlabel('Sequence Length (aa)', fontsize=11)
ax.set_ylabel('MLP Accuracy - Linear Accuracy', fontsize=11)
ax.set_title('C. Nonlinearity Gap (best MLP - Linear)', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

fig.suptitle('ProtMamba Nonlinear Probe Validation: Is the Encoding Linear or Curved?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/nonlinear_probe_comparison.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS_DIR}/nonlinear_probe_comparison.png')


In [ ]:
# Summary Heatmap

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax_idx, pool_mode in enumerate(['local', 'global']):
    ax = axes[ax_idx]
    df_pool = df_probes[df_probes['pool_mode'] == pool_mode].sort_values('target_length')
    if df_pool.empty:
        continue

    heatmap_data = pd.DataFrame({
        'Linear': df_pool['linear_mean'].values,
        'MLP (256,64)': df_pool['mlp_mean'].values,
        'MLP (512,256,64)': df_pool['mlp_wide_mean'].values,
    }, index=[f'L={int(L)}' for L in df_pool['target_length']])

    sns.heatmap(heatmap_data.T, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0.5, vmin=0.4, vmax=1.0, ax=ax,
                linewidths=1, linecolor='white',
                cbar_kws={'label': 'Accuracy'})
    label = 'A' if ax_idx == 0 else 'B'
    ax.set_title(f'{label}. {pool_mode.title()} Pooling',
                 fontweight='bold', fontsize=12)
    ax.set_ylabel('Probe Type')
    ax.set_xlabel('Sequence Length')

fig.suptitle('Probe Accuracy Heatmap: Green = Signal Recovered, Red = At Chance',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/nonlinear_probe_heatmap.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS_DIR}/nonlinear_probe_heatmap.png')


In [ ]:
# Side-by-Side with Ghost Detection Results

ghost_csv = './results/protmamba_ghost_detection/results/ghost_frozen_head.csv'

print('=' * 70)
print('COMPARISON: Original Ghost Detection vs. Nonlinear Probe')
print('=' * 70)

if os.path.exists(ghost_csv):
    df_ghost = pd.read_csv(ghost_csv)
    print('\nOriginal Ghost Detection (linear probe only):')
    print(df_ghost.to_string(index=False))

    print('\nNew results (linear + MLP probes, local pooling):')
    df_local = df_probes[df_probes['pool_mode'] == 'local'].copy()
    if not df_local.empty:
        summary = df_local[['target_length', 'signal_fraction',
                            'linear_mean', 'mlp_mean', 'mlp_wide_mean',
                            'mlp_minus_linear', 'mlp_wide_minus_linear']].copy()
        summary.columns = ['Length', 'Signal%', 'Linear', 'MLP', 'MLP-Wide',
                           'Gap(MLP)', 'Gap(Wide)']
        print(summary.to_string(index=False))

        print('\nKey comparison:')
        for _, row in df_local.iterrows():
            L = int(row['target_length'])
            lin = row['linear_mean']
            best_mlp = max(row['mlp_mean'], row['mlp_wide_mean'])
            print(f'  L={L}: Linear={lin:.3f}, Best MLP={best_mlp:.3f}')
            if lin <= 0.55 and best_mlp > 0.55:
                print(f'         -> REVISED: Not uninformative but nonlinearly encoded')
            elif lin <= 0.55 and best_mlp <= 0.55:
                print(f'         -> CONFIRMED: Genuinely uninformative at this length')
            else:
                print(f'         -> Signal accessible to both probe types')
else:
    print(f'\nGhost Detection results not found at {ghost_csv}')
    print('Run ProtMamba_Ghost_Detection.ipynb first for comparison.')
    print('\nStandalone results:')
    print(df_probes[['target_length', 'pool_mode', 'linear_mean',
                     'mlp_mean', 'mlp_wide_mean']].to_string(index=False))
